# Inference

In [1]:
import torch
from olmoearth_pretrain.model_loader import ModelID, load_model_from_id
from olmoearth_pretrain.datatypes import MaskedOlmoEarthSample, MaskValue

# Load model from HuggingFace
model = load_model_from_id(ModelID.OLMOEARTH_V1_BASE)
model.eval()

/home/users_model/miniconda3/envs/olmoearth/lib/python3.12/importlib/__init__.py:90: FutureWarning: The 'helios' package has been renamed to 'olmoearth_pretrain'. Please update your imports; this compatibility shim will be removed in a future release.
  return _bootstrap._gcd_import(name[level:], package, level)


LatentMIM(
  (encoder): Encoder(
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (q): Linear(in_features=768, out_features=768, bias=True)
          (k): Linear(in_features=768, out_features=768, bias=True)
          (v): Linear(in_features=768, out_features=768, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path): DropPath()
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (drop1): Dropout(p=0.0, inplace=False)
          (fc2): Linear(in_features=3072, out_feature

In [3]:
# Create synthetic input (B=1, H=64, W=64, T=1, C=12 for Sentinel-2)
dummy_image = torch.randn(1, 64, 64, 1, 12)
dummy_mask = torch.ones(1, 64, 64, 1, 3) * MaskValue.ONLINE_ENCODER.value
dummy_timestamps = torch.tensor([[[15, 6, 2024]]])  # day, month (0-indexed), year

sample = MaskedOlmoEarthSample(
    sentinel2_l2a=dummy_image,
    sentinel2_l2a_mask=dummy_mask,
    timestamps=dummy_timestamps,
)

with torch.no_grad():
    output = model.encoder(sample, fast_pass=True, patch_size=4)
    features = output["tokens_and_masks"].sentinel2_l2a
    print(f"Output shape: {features.shape}")  # (B, H', W', T, S, D)

Output shape: torch.Size([1, 16, 16, 1, 3, 768])


In [1]:
import glob
import numpy as np
import rasterio
from olmoearth_pretrain.data.constants import Modality
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

In [2]:
# Get the JP2 filenames that we need to read, in the band order expected by OlmoEarth.
fnames = []
for band_name in Modality.SENTINEL2_L2A.band_order:
    fname = glob.glob(f"/mnt/ht2-nas2/00-model/00-jiangzf/coderepo/*.SAFE/GRANULE/*/IMG_DATA/*/*_{band_name}_*.jp2")[0]
    fnames.append(fname)

# Get the CRS and transform from the first band, which is B02.
with rasterio.open(fnames[0]) as src:
    crs = src.crs
    transform = src.transform
    width = src.width
    height = src.height

# We limit the width/height to 512x512 in case there is limited memory.
width = 512
height = 512

# Now read all of the bands.
image = np.zeros((len(fnames), height, width), dtype=np.int32)
for band_idx, fname in enumerate(fnames):
    with rasterio.open(fname) as src:
        with rasterio.vrt.WarpedVRT(
            src,
            crs=crs,
            transform=transform,
            width=width,
            height=height,
            resampling=Resampling.bilinear,
        ) as vrt:
            image[band_idx, :, :] = vrt.read(1)

# Rearrange to BHWTC.
image = image.transpose(1, 2, 0)[None, :, :, None, :]

In [5]:
from olmoearth_pretrain.data.normalize import Normalizer, Strategy

normalizer = Normalizer(Strategy.COMPUTED)
image = normalizer.normalize(Modality.SENTINEL2_L2A, image)

In [6]:
import torch
from olmoearth_pretrain.model_loader import ModelID, load_model_from_id
from olmoearth_pretrain.datatypes import MaskedOlmoEarthSample, MaskValue

model = load_model_from_id(ModelID.OLMOEARTH_V1_BASE)
device = torch.device("cuda")
model.to(device)

# Run the model on the topleft 64x64 of the image.
sample = MaskedOlmoEarthSample(
    sentinel2_l2a=torch.tensor(image[:, 0:64, 0:64, :, :], dtype=torch.float32, device=device),
    # The mask shape is BHWTS, where S is the number of band sets (3 for Sentinel-2).
    sentinel2_l2a_mask=torch.ones((1, 64, 64, 1, 3), dtype=torch.float32, device=device) * MaskValue.ONLINE_ENCODER.value,
    # The timestamps is (day of month 1-31, month 0-11, year).
    # The values here correspond to the date of our sample Sentinel-2 image of Seattle
    # (2025-08-22).
    timestamps=torch.tensor([22, 7, 2025], device=device)[None, None, :],
)
tokens_and_masks = model.encoder(
    sample, fast_pass=True, patch_size=4,
)["tokens_and_masks"]
# Get the Sentinel-2 features.
modality_features = tokens_and_masks.sentinel2_l2a
# Pool the features over the timestep and band set dimensions so we end up with a BHWC
# feature map.
pooled = modality_features.mean(dim=[3, 4])

/home/users_model/miniconda3/envs/olmoearth/lib/python3.12/importlib/__init__.py:90: FutureWarning: The 'helios' package has been renamed to 'olmoearth_pretrain'. Please update your imports; this compatibility shim will be removed in a future release.
  return _bootstrap._gcd_import(name[level:], package, level)


In [ ]:
import torch
import torch.optim as optim
from olmoearth_pretrain.model_loader import ModelID, load_model_from_id
from olmoearth_pretrain.datatypes import MaskedOlmoEarthSample, MaskValue

# 1. Setup Model, Optimizer, and Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model_from_id(ModelID.OLMOEARTH_V1_BASE)
model.to(device)
model.train()  # Set model to training mode

# Define optimizer (learning rate should be tuned based on batch size and schedule)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

# 2. Prepare Dummy Data (Using the same structure as your inference example)
# Note: For training, you usually need both the input and the target. 
# The LatentMIM forward pass typically handles the masking and target generation internally 
# or expects specific inputs depending on the implementation.
# Here we assume the standard MIM flow where the model computes loss internally 
# or we compute it based on encoder/decoder outputs.

# Create a simple dummy batch
B, H, W, T, C = 1, 64, 64, 1, 12
dummy_image = torch.randn(B, H, W, T, C, device=device)
dummy_mask = torch.ones(B, H, W, T, 3, device=device) * MaskValue.ONLINE_ENCODER.value
dummy_timestamps = torch.tensor([[15, 6, 2024]], device=device)[None, :] # Shape: B, 1, 3

sample = MaskedOlmoEarthSample(
    sentinel2_l2a=dummy_image,
    sentinel2_l2a_mask=dummy_mask,
    timestamps=dummy_timestamps,
)

# 3. Minimal Training Loop
num_epochs = 5
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Forward pass
    # Note: Check the specific return type of your model's forward pass.
    # Some MIM models return a loss directly, others return predictions.
    # Based on the previous cell, model.encoder returns tokens. 
    # Usually, training involves the full model (encoder + decoder).
    
    # Assuming the model has a forward method that computes loss for MIM
    # If not, you might need to call encoder and decoder separately and compute MSE.
    # Below is a generic pattern. You might need to adjust based on olmoearth_pretrain API.
    
    try:
        # Try calling the model directly if it supports end-to-end training loss
        output = model(sample)
        loss = output.loss if hasattr(output, 'loss') else output
    except Exception as e:
        # Fallback: Manual forward pass if the model doesn't return loss directly
        # This part depends heavily on how LatentMIM is implemented in olmoearth_pretrain
        # Typically: 
        # 1. Encode visible patches
        # 2. Decode to predict masked patches
        # 3. Compute loss against target encoder outputs
        
        # For this minimal example, let's assume we just minimize the norm of features 
        # as a placeholder if the actual loss function isn't easily accessible.
        # REPLACE THIS WITH ACTUAL LOSS CALCULATION
        with torch.no_grad():
            target_output = model.target_encoder(sample, fast_pass=True, patch_size=4)
            
        pred_output = model.decoder(
            model.encoder(sample, fast_pass=True, patch_size=4)["tokens_and_masks"],
            sample
        )
        
        # Simple MSE loss between prediction and target (shape alignment needed)
        # This is illustrative; check actual tensor shapes from model outputs
        loss = torch.nn.functional.mse_loss(pred_output, target_output["tokens_and_masks"].sentinel2_l2a)

    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

print("Training loop finished.")

0

In [ ]:

"""Trying to prototype fitting everything into olmo core."""

import sys
import os
# Adjust the path below to point to the root of your project where 'script.py' is located
# Assuming the notebook is in olmoearth_pretrain-main/scripts/jzf/
# and script.py is in olmoearth_pretrain-main/
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../scripts/official'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


import logging

from script import (
    build_common_components,
    build_dataloader_config,
    build_dataset_config,
    build_train_module_config,
    build_trainer_config,
)

from olmoearth_pretrain.internal.experiment import CommonComponents, main
from olmoearth_pretrain.internal.utils import MODEL_SIZE_ARGS
from olmoearth_pretrain.nn.flexihelios import (
    EncoderConfig,
    PredictorConfig,
)
from olmoearth_pretrain.nn.latent_mim import LatentMIMConfig

logger = logging.getLogger(__name__)

MAX_PATCH_SIZE = 8
MIN_PATCH_SIZE = 1

import os

def build_model_config(common: CommonComponents) -> LatentMIMConfig:
    """Build the model config for an experiment."""
    model_size = MODEL_SIZE_ARGS["nano"]

    encoder_config = EncoderConfig(
        embedding_size=model_size["encoder_embedding_size"],
        num_heads=model_size["encoder_num_heads"],
        depth=model_size["encoder_depth"],
        mlp_ratio=model_size["mlp_ratio"],
        supported_modality_names=common.training_modalities,
        max_patch_size=MAX_PATCH_SIZE,
        drop_path=0.1,
        max_sequence_length=12,
        use_linear_patch_embed=False,
    )
    decoder_config = PredictorConfig(
        encoder_embedding_size=model_size["encoder_embedding_size"],
        decoder_embedding_size=model_size["decoder_embedding_size"],
        depth=model_size["decoder_depth"],
        mlp_ratio=model_size["mlp_ratio"],
        num_heads=model_size["decoder_num_heads"],
        supported_modality_names=common.training_modalities,
        max_sequence_length=12,
    )
    model_config = LatentMIMConfig(
        encoder_config=encoder_config,
        decoder_config=decoder_config,
    )
    return model_config


# if __name__ == "__main__":
#     main(
#         common_components_builder=build_common_components,
#         model_config_builder=build_model_config,
#         train_module_config_builder=build_train_module_config,
#         dataset_config_builder=build_dataset_config,
#         dataloader_config_builder=build_dataloader_config,
#         trainer_config_builder=build_trainer_config,
#     )


if __name__ == "__main__":
    import sys
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"
    
    # 调试模式：硬编码参数
    debug_mode = True  # 调试时设为 True，正常训练时改为 False
    
    if debug_mode:
        # 直接设置 sys.argv
        sys.argv = [
            "scripts/official/nano.py",
            "train_single",
            "debug_run_new", 
            "local",
            "--dataset.h5py_dir=/mnt/ht2-nas2/QH_Group/H5_DIR/h5py_data_w_missing_timesteps_zstd_3_128_x_4/cdl_landsat_openstreetmap_raster_sentinel1_sentinel2_l2a_srtm_worldcereal_worldcover_wri_canopy_height_map/3996",
            "--data_loader.global_batch_size=640",
            "--trainer.max_duration.value=1",
            "--data_loader.num_workers=0",
            "--trainer.callbacks.wandb.enabled=False",
        ]
    
    main(
        common_components_builder=build_common_components,
        model_config_builder=build_model_config,
        train_module_config_builder=build_train_module_config,
        dataset_config_builder=build_dataset_config,
        dataloader_config_builder=build_dataloader_config,
        trainer_config_builder=build_trainer_config,
    )


/home/users_model/miniconda3/envs/olmoearth/lib/python3.12/site-packages/class_registry/entry_points.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import iter_entry_points
/home/users_model/miniconda3/envs/olmoearth/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/users_mode

2026-04-19 10:43:38.981	iZ1pp00ofwvuxcw0wxchchZ:0	olmo_core.utils:227	INFO	Setting env var 'OMP_NUM_THREADS' to '8'
2026-04-19 10:43:38.982	iZ1pp00ofwvuxcw0wxchchZ:0	olmo_core.utils:227	INFO	Setting env var 'TOKENIZERS_PARALLELISM' to 'false'
2026-04-19 10:43:38.983	iZ1pp00ofwvuxcw0wxchchZ:0	olmoearth_pretrain.internal.experiment:199	INFO	Common overrides: []
2026-04-19 10:43:38.993	iZ1pp00ofwvuxcw0wxchchZ:0	olmoearth_pretrain.internal.experiment:201	INFO	Common: CommonComponents(run_name='debug_run_new', save_folder='./local_output/checkpoints/anonymous/debug_run_new', training_modalities=['sentinel2_l2a', 'sentinel1', 'landsat', 'worldcover', 'srtm', 'openstreetmap_raster', 'wri_canopy_height_map', 'cdl', 'worldcereal'], launch=None, nccl_debug=False, tokenization_config=None)
2026-04-19 10:43:38.996	iZ1pp00ofwvuxcw0wxchchZ:0	olmoearth_pretrain.internal.experiment:220	INFO	Overrides: ['--dataset.h5py_dir=/mnt/ht2-nas2/QH_Group/H5_DIR/h5py_data_w_missing_timesteps_zstd_3_128_x_4/cdl_l